In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing.validate_alignment import RasterAlignmentValidator

In [2]:
RASTER_PATHS = {
    's2': '/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2020_partidos_ambas.tif',
    'morphometric': '/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2025_building_density_updated.tif',
    'reference_labels': '/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/reference_labels_partidos_AMBA.tif'
}

In [3]:
OUTPUT_DIR = '/content/drive/MyDrive/Fine-Tuning-Workflow/Preprocessing'

In [4]:
# Configuración
VERBOSE = True  # Mostrar información detallada

print("\n🔧 Configuración final:")
for name, path in RASTER_PATHS.items():
    print(f"   {name}: {path}")
print(f"   Output: {OUTPUT_DIR}")


🔧 Configuración final:
   s2: /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2020_partidos_ambas.tif
   morphometric: /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2025_building_density_updated.tif
   reference_labels: /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/reference_labels_partidos_AMBA.tif
   Output: /content/drive/MyDrive/Fine-Tuning-Workflow/Preprocessing


In [5]:
print("\n📂 Verificando existencia de archivos...")

missing_files = []
existing_files = []

for name, path in RASTER_PATHS.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"   ✅ {name}: {os.path.basename(path)} ({size_mb:.1f} MB)")
        existing_files.append(path)
    else:
        print(f"   ❌ {name}: ARCHIVO NO ENCONTRADO - {path}")
        missing_files.append((name, path))

if missing_files:
    print(f"\n⚠️ Se encontraron {len(missing_files)} archivos faltantes.")
    print("   📝 Tip para Colab: Verifica las rutas usando el panel de archivos")
    print("   🔍 Rutas comunes en Colab:")
    print("      - Archivos subidos: /content/filename.tif")
    print("      - Google Drive: /content/drive/MyDrive/path/filename.tif")
    print("      - Kaggle datasets: /kaggle/input/dataset/filename.tif")
else:
    print(f"\n✅ Todos los archivos ({len(existing_files)}) están disponibles.")



📂 Verificando existencia de archivos...
   ✅ s2: s2_2020_partidos_ambas.tif (2949.9 MB)
   ✅ morphometric: s2_2025_building_density_updated.tif (1275.1 MB)
   ✅ reference_labels: reference_labels_partidos_AMBA.tif (5.9 MB)

✅ Todos los archivos (3) están disponibles.


In [6]:

if not missing_files:
    print("\n🚀 Iniciando validación de alineación...")
    print("   ⏱️  Nota: Esto puede tardar unos minutos dependiendo del tamaño de archivos")
    print("   📊 Progreso visible en tiempo real...")

    # Crear validador
    validator = RasterAlignmentValidator(verbose=VERBOSE)

    # Ejecutar validación
    try:
        success = validator.validate_rasters(list(RASTER_PATHS.values()))

        # Mostrar resultado final
        if success:
            print("\n🎉 RESULTADO: Todos los rasters están correctamente alineados")
            print("   ➡️ Puedes proceder con el grid sampling y extracción de patches")
        else:
            print("\n⚠️ RESULTADO: Se encontraron problemas de alineación")
            print("   ➡️ Revisa las recomendaciones arriba antes de continuar")

        # Guardar reporte detallado
        report_path = os.path.join(OUTPUT_DIR, 'alignment_report.json')
        validator.save_report(report_path)
        print(f"\n📄 Reporte detallado guardado en: {report_path}")
        print("💡 Tip: Puedes descargar el reporte desde el panel de archivos")

    except Exception as e:
        print(f"\n❌ Error durante la validación: {str(e)}")
        print("   💡 Verifica que los archivos sean rasters válidos")
        print("   🔧 Puedes reiniciar el runtime si es necesario")

else:
    print("\n❌ No se puede ejecutar la validación hasta que todos los archivos estén disponibles.")
    print("\n📋 Pasos para subir archivos a Colab:")
    print("1. Opción Drive: Subir a Google Drive y montar")
    print("2. Opción Local: Arrastrar archivos al panel lateral")
    print("3. Opción Kaggle: Conectar dataset de Kaggle")


🚀 Iniciando validación de alineación...
   ⏱️  Nota: Esto puede tardar unos minutos dependiendo del tamaño de archivos
   📊 Progreso visible en tiempo real...
[INFO] 🚀 Iniciando validación de 3 rasters...
[INFO] ✅ CRS consistente: EPSG:4326
[INFO] ✅ Resolución consistente: 0.000090 x 0.000090
[INFO] ✅ Bounds se superponen correctamente
[INFO]    Intersección: 1.67 x 1.62 unidades
[INFO] ✅ Grids de píxeles correctamente alineados
[INFO] 
🎉 ¡Todos los rasters están perfectamente alineados!

🔍 REPORTE DE VALIDACIÓN DE ALINEACIÓN DE RASTERS

📁 Rasters analizados: 3
   • s2_2020_partidos_ambas.tif (2949.9 MB)
   • s2_2025_building_density_updated.tif (1275.1 MB)
   • reference_labels_partidos_AMBA.tif (5.9 MB)

📊 Estado general: PERFECT_ALIGNMENT

✅ No se encontraron problemas de alineación


🎉 RESULTADO: Todos los rasters están correctamente alineados
   ➡️ Puedes proceder con el grid sampling y extracción de patches
[INFO] 📝 Reporte guardado en: /content/drive/MyDrive/Fine-Tuning-Workflo

In [7]:

print("\n" + "="*60)
print("📋 ANÁLISIS DETALLADO")
print("="*60)

if not missing_files:
    print("\n🔍 Información detallada de cada raster:")

    validator_detailed = RasterAlignmentValidator(verbose=False)

    for name, path in RASTER_PATHS.items():
        print(f"\n📊 {name.upper()}:")

        try:
            info = validator_detailed.get_raster_info(path)

            if info:
                print(f"   Archivo: {info['filename']}")
                print(f"   CRS: {info['crs']}")
                print(f"   Dimensiones: {info['width']} x {info['height']} pixels")
                print(f"   Bandas: {info['count']}")
                print(f"   Tipo de dato: {info['dtype']}")
                print(f"   Resolución: {info['pixel_size_x']:.6f} x {info['pixel_size_y']:.6f}")
                print(f"   Bounds: {info['bounds']}")
                print(f"   NoData: {info['nodata']}")
                print(f"   Tamaño: {info['size_mb']:.1f} MB")

                if info['area_m2']:
                    area_km2 = info['area_m2'] / 1_000_000
                    print(f"   Área cubierta: {area_km2:.2f} km²")

                # Información específica para archivos grandes
                if info['size_mb'] > 500:
                    print(f"   ⚠️  Archivo grande ({info['size_mb']:.1f} MB) - procesamiento puede ser lento")

        except Exception as e:
            print(f"   ❌ Error leyendo {name}: {str(e)}")



📋 ANÁLISIS DETALLADO

🔍 Información detallada de cada raster:

📊 S2:
   Archivo: s2_2020_partidos_ambas.tif
   CRS: EPSG:4326
   Dimensiones: 18590 x 17979 pixels
   Bandas: 10
   Tipo de dato: float32
   Resolución: 0.000090 x 0.000090
   Bounds: BoundingBox(left=-59.37819112265831, bottom=-35.42146996811685, right=-57.708223009480115, top=-33.80638891879836)
   NoData: -inf
   Tamaño: 2949.9 MB
   ⚠️  Archivo grande (2949.9 MB) - procesamiento puede ser lento

📊 MORPHOMETRIC:
   Archivo: s2_2025_building_density_updated.tif
   CRS: EPSG:4326
   Dimensiones: 18590 x 17979 pixels
   Bandas: 1
   Tipo de dato: float32
   Resolución: 0.000090 x 0.000090
   Bounds: BoundingBox(left=-59.37819112265831, bottom=-35.42146996811685, right=-57.708223009480115, top=-33.80638891879836)
   NoData: None
   Tamaño: 1275.1 MB
   ⚠️  Archivo grande (1275.1 MB) - procesamiento puede ser lento

📊 REFERENCE_LABELS:
   Archivo: reference_labels_partidos_AMBA.tif
   CRS: EPSG:4326
   Dimensiones: 18590 x 

# Reprojection

In [ ]:
!pip install rasterio
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.enums import Resampling

# Archivos
input_file = '/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/CREODIAS_Direct/s2_mosaic_multiple_tiles.tif'
output_file = '/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/CREODIAS_Direct/s2_mosaic_multiple_tiles_aligned.tif'
reference_file = '/content/drive/MyDrive/ML_Inference_Data/morphometric_data/building_density.tif'  # Ajusta la ruta

print("📍 Leyendo archivo de referencia para alineamiento...")

# Abrir archivo de referencia para obtener parámetros espaciales
with rasterio.open(reference_file) as ref:
    target_crs = ref.crs
    target_transform = ref.transform
    target_width = ref.width
    target_height = ref.height
    target_bounds = ref.bounds

    print(f"   CRS objetivo: {target_crs}")
    print(f"   Resolución objetivo: {ref.res}")
    print(f"   Dimensiones objetivo: {target_width} x {target_height}")

print("\n🔄 Reproyectando y alineando archivo S2...")

# Reproyectar y alinear en un solo paso
with rasterio.open(input_file) as src:
    print(f"   CRS original: {src.crs}")
    print(f"   Dimensiones originales: {src.width} x {src.height}")

    # Copiar metadata y actualizar con parámetros del archivo de referencia
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': target_crs,
        'transform': target_transform,
        'width': target_width,
        'height': target_height
    })

    # Crear archivo de salida y reproyectar todas las bandas
    with rasterio.open(output_file, 'w', **kwargs) as dst:
        for i in range(1, src.count + 1):
            print(f"   Procesando banda {i}/{src.count}...")
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=target_transform,
                dst_crs=target_crs,
                resampling=Resampling.bilinear
            )

print(f"\n✅ Reproyección y alineamiento completos!")
print(f"📁 Archivo guardado: {output_file}")

# Verificar el archivo final
with rasterio.open(output_file) as final:
    print(f"\n📊 Archivo final:")
    print(f"   CRS: {final.crs}")
    print(f"   Dimensiones: {final.width} x {final.height}")
    print(f"   Resolución: {final.res}")
    print(f"   Bounds: {final.bounds}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 55.3 MB/s eta 0:00:00
📍 Leyendo archivo de referencia para alineamiento...
   CRS objetivo: EPSG:4326
   Resolución objetivo: (8.983152841195215e-05, 8.983152841195215e-05)
   Dimensiones objetivo: 11563 x 11237

🔄 Reproyectando y alineando archivo S2...
   CRS original: EPSG:32721
   Dimensiones originales: 10980 x 10980
   Procesando banda 1/10...
   Procesando banda 2/10...
   Procesando banda 3/10...
   Procesando banda 4/10...
   Procesando banda 5/10...
   Procesando banda 6/10...
   Procesando banda 7/10...
   Procesando banda 8/10...
   Procesando banda 9/10...
   Procesando banda 10/10...

✅ Reproyección y alineamiento completos!
📁 Archivo guardado: /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/CREODIAS_Direct/s2_mosaic_multiple_tiles_aligned.tif

📊 Archivo final:
   CRS: EPSG:4326
   Dimensiones: 11563 x 11237
   Resolución: (8.983152841195215e-05, 8.983152841195215e-05)
   Bounds: BoundingBox(left=-59.056324756